In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
torch.manual_seed(42)
np.random.seed(42)

In [3]:
data = load_breast_cancer()
X = data.data
y = data.target

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
# Convert to PyTorch tensors
# BCEWithLogitsLoss expects float targets
# often shaped as (N, 1)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

In [7]:
# Create DataLoaders

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [8]:
# Simple MLP for tabular data

class BreastCancerNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)   # one output logit for binary classification
        )

    def forward(self, x):
        return self.model(x)

In [9]:
model = BreastCancerNet(input_dim=X_train.shape[1])

In [10]:
# Loss and optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [11]:
# Training loop

num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_X, batch_y in train_loader:
        # Forward pass
        logits = model(batch_X) # it calls model.forward(batch_X)
        loss = criterion(logits, batch_y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}")

Epoch [10/50] - Loss: 0.1618
Epoch [20/50] - Loss: 0.0713
Epoch [30/50] - Loss: 0.0501
Epoch [40/50] - Loss: 0.0408
Epoch [50/50] - Loss: 0.0350


In [12]:
# Evaluation

model.eval()

all_probs = []
all_preds = []
all_true = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        logits = model(batch_X)
        probs = torch.sigmoid(logits)              # convert logits to probabilities
        preds = (probs >= 0.5).float()            # threshold at 0.5

        all_probs.extend(probs.squeeze().cpu().numpy())
        all_preds.extend(preds.squeeze().cpu().numpy())
        all_true.extend(batch_y.squeeze().cpu().numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)

acc = accuracy_score(all_true, all_preds)
print("\nTest accuracy:", acc)

print("\nClassification report:")
print(classification_report(all_true, all_preds, target_names=data.target_names))

print("\nConfusion matrix:")
print(confusion_matrix(all_true, all_preds))


Test accuracy: 0.956140350877193

Classification report:
              precision    recall  f1-score   support

   malignant       0.91      0.98      0.94        42
      benign       0.99      0.94      0.96        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114


Confusion matrix:
[[41  1]
 [ 4 68]]
